# Per-range statistics: one wide table and an across-range summary

Every per-range number in one place. This notebook extends the per-range metrics table of
`0_aggregate_by_mountain_range.ipynb` (identity, median onset, MAD, lapse rates) with everything derived from the
same cubes: mapped-snow and terrain area, the inclusion rule, the cosine sunny–shady difference, per-year coverage,
the interannual variability of the anomaly and of the topographic controls, their co-variation, the
spring-temperature sensitivity with both estimators and a permutation test, and the sensitivity expected from the
annual temperature cycle (Evan & Eisenman 2021). The functions are in `gsro_analysis.range_metrics`.

```
reads                                                       writes (results/<version>/)
─────                                                       ──────────────────────────
data/aggregation/<v>/all_mountain_ranges_fcf_lte_50.nc ──┐   mountain_range_statistics.csv             one row per range
                                                         │   mountain_range_statistics_dictionary.csv  every column: unit, meaning
data/aggregation/<v>/sparse_mountain_ranges.parquet ─────┤   mountain_range_hypsometry.csv             km² per range x 100 m band
results/<v>/mountain_range_metrics.csv ──────────────────┼─► mountain_range_band_aspect.csv           per band: onset, aspect amplitude
                                                         │   mountain_range_yearly_topography.csv      per range-year: lapse, sunny-shady, aspect
GMBA v2 polygons (web) ──────────────────────────────────┤   mountain_range_summary.json               across-range numbers
climate/temperature_sensitivity_comparison/data/          │   mountain_range_inclusion_robustness.csv   summary under other inclusion rules
  evan_eisenman_2021_era5_T0_T1_climatology.nc (optional) ┘
```

Run `0_aggregate_by_mountain_range.ipynb` first; the Evan & Eisenman columns are filled when that comparison
notebook has cached its ERA5 climatology. Across-range numbers are counts and medians of per-range values, never
pooled fits.

In [ ]:
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr

from gsro_analysis import aggregate, paths, range_metrics as rm, results, settings

In [ ]:
config = settings.load_config()
VERSION = config.version
WATER_YEARS = [int(y) for y in config.water_years]
UNIT = 'mountain_ranges'
aggregation_dir = paths.aggregation_dir(UNIT, VERSION)
results_dir = paths.resultsdir(UNIT, VERSION)

INCLUSION_SNOW_PCT = rm.INCLUSION_SNOW_PCT       # mapped snow (FCF <= 50 %) must cover at least this % of the GMBA polygon
ROBUSTNESS_RULES = [0, 1, 2, 3, 5]                # the summary is repeated under these inclusion thresholds (%)
FOREST_CLASSES = list(range(5, 10))               # FCF classes (50, 60] ... (90, 100]: "snow under forest"
N_PERM_RANGE = 10000                              # permutations per range (the per-range permutation p)
N_PERM_FIELD = 1000                               # shared-year permutations (field significance)
N_PERM_COVARIATION = 2000                         # within-range shuffles (lapse-aspect co-variation null)
SEED = 0
ee_climatology_path = paths.ROOT / 'analyses/climate/temperature_sensitivity_comparison/data/evan_eisenman_2021_era5_T0_T1_climatology.nc'
print(f'{VERSION} | water years {WATER_YEARS[0]}-{WATER_YEARS[-1]} | inclusion: snow >= {INCLUSION_SNOW_PCT} % of the polygon')

## 1. Inputs

In [ ]:
open_ds = xr.open_dataset(aggregation_dir / 'all_mountain_ranges_fcf_lte_50.nc')     # the analyses' cube (FCF <= 50 %)
open_ds

In [ ]:
metrics_df = pd.read_csv(results_dir / 'mountain_range_metrics.csv').set_index('name')   # from 0_aggregate_by_mountain_range
metrics_df

In [ ]:
gmba_gdf = gpd.read_file('zip+' + settings.GMBA_URL)        # GMBA Inventory v2.0, standard 300 (Snethlage et al. 2022)
gmba_gdf = gmba_gdf[gmba_gdf['GMBA_V2_ID'].isin(metrics_df['GMBA_V2_ID'])][['GMBA_V2_ID', 'geometry']]
gmba_gdf

In [ ]:
# elevation-only views of the sparse table for the hypsometry: all terrain (every forest class, snow-free ranges
# kept), analysed snow under the analyses' filter, and analysed snow under forest (FCF > 50 %)
sparse_path = aggregation_dir / f'sparse_{UNIT}.parquet'
terrain_ds = aggregate.view(aggregate.load_sparse(sparse_path, ('elevation',)), UNIT, ('elevation',), WATER_YEARS,
                            fcf=None, analysed_units_only=False)
open_elev_ds = aggregate.view(aggregate.load_sparse(sparse_path, ('elevation',), fcf=aggregate.FILTER_VIEWS['fcf_lte_50']),
                              UNIT, ('elevation',), WATER_YEARS, fcf=aggregate.FILTER_VIEWS['fcf_lte_50'])
forest_elev_ds = aggregate.view(aggregate.load_sparse(sparse_path, ('elevation',), fcf=FOREST_CLASSES),
                                UNIT, ('elevation',), WATER_YEARS, fcf=FOREST_CLASSES)
terrain_ds

In [ ]:
ee_clim_ds = xr.open_dataset(ee_climatology_path) if ee_climatology_path.exists() else None   # ERA5 T0 / T1 (0.7 deg)
print('Evan & Eisenman climatology:', 'found' if ee_clim_ds is not None else f'missing ({ee_climatology_path.name}): run that notebook first')
ee_clim_ds

## 2. Area, mapped snow and the inclusion rule

Mapped snow is the area of 80 m pixels with an 11-year median onset under the analyses' filter (FCF ≤ 50 %); it is
not snow-covered area. A range is analysed when it has metrics and its mapped snow covers at least
`INCLUSION_SNOW_PCT` % of its GMBA polygon (EPSG:6933 area): below that the ranges are tropical or barely snowy and
their statistics rest on a few hundred km².

In [ ]:
# km² per range and 100 m band: all terrain, analysed snow (FCF <= 50 %) and analysed snow under forest (FCF > 50 %)
id_to_name = dict(zip(open_ds['GMBA_V2_ID'].values.astype(int), open_ds['mountain_range'].values.astype(str)))
names_of = lambda ds: ds.assign_coords(mountain_range=[id_to_name.get(int(i)) for i in ds['mountain_range'].values])
keep = lambda ds: names_of(ds).sel(mountain_range=[id_to_name.get(int(i)) is not None for i in ds['mountain_range'].values])
hypsometry_df = rm.hypsometry(keep(terrain_ds), keep(open_elev_ds), keep(forest_elev_ds))
hypsometry_df.groupby('name')[['terrain_km2', 'snow_km2', 'snow_under_forest_km2']].sum().describe()

In [ ]:
snow_df = rm.area_totals(hypsometry_df).reindex(metrics_df.index)        # flat pixels included (elevation-only views)
polygon_km2 = rm.polygon_area_km2(gmba_gdf)
snow_df['polygon_km2'] = metrics_df['GMBA_V2_ID'].map(polygon_km2).values
has_metrics = metrics_df['mean_median_onset_dowy'].notna()
snow_df = snow_df.join(rm.inclusion(snow_df['snow_km2'], snow_df['polygon_km2'], has_metrics, INCLUSION_SNOW_PCT))
print(f"{int(has_metrics.sum())} ranges with metrics, {int(snow_df['included'].sum())} included")
snow_df.sort_values('snow_pct_of_polygon')

## 3. Coverage, range-year anomalies and spring temperature

The metrics table's view: bins with at least `rm.MIN_PIXELS_PER_BIN` pixels; a range-year counts when at least
`rm.MIN_YEAR_FRACTION` of the range's median-onset pixels have a valid onset that year. Coverage is stored for every
range-year and range-years below `rm.LOW_COVERAGE` are flagged (Sentinel-1B was lost in December 2021, which thins
WY2022–2025 at high latitudes).

In [ ]:
view_ds = rm.base_view(open_ds)
coverage_df = rm.coverage(view_ds)
anomaly_df = rm.range_year_anomaly(view_ds)
temperature_df = rm.spring_temperature(view_ds)
print(f'{int(anomaly_df.notna().sum().sum())} range-years with an anomaly, '
      f'{int(((coverage_df < rm.LOW_COVERAGE) & anomaly_df.notna()).sum().sum())} of them below {rm.LOW_COVERAGE:.0%} coverage')
anomaly_df.round(1)

In [ ]:
interannual_df = rm.interannual_anomaly(anomaly_df)
interannual_df.describe()

## 4. Topography: the sunny–shady difference, per-year lapse rate and aspect amplitude, and their co-variation

The sunny–shady difference: per range, onset per 100 m band × 15° aspect bin minus the band's median across
aspects (elevation removed), averaged over bands weighted by pixel count; then the cosine
`onset = a + A cos(aspect − φ)` is fitted and the difference is **2A** days, with φ + 180° the earliest-melting
direction. The same fit per water year, and the per-year lapse rate (pixel-weighted regression of the aspect-mean
onset on elevation, at least 5 bands over 500 m), keep a bin-year only above `rm.MIN_BIN_YEAR_FRACTION` of the bin's
median-onset pixels. Their within-range co-variation is the Pearson r across years (at least
`rm.MIN_YEARS_INTERANNUAL` years).

In [ ]:
sunny_shady_df, sunny_shady_year_df, earliest_aspect_year_df = rm.sunny_shady(view_ds)
sunny_shady_df.describe()

In [ ]:
lapse_year_df, _ = rm.yearly_topography(view_ds)
covariation_df = rm.topography_covariation(lapse_year_df, sunny_shady_year_df)
covariation_df.describe()

## 5. Spring-temperature sensitivity

The range-year anomaly against the range's mean spring (first three melt-season months) 2 m temperature anomaly from
ERA5-Land, over the paired years: OLS slope with its t-based 95 % CI, Pearson r and p (the primary estimator); the
Theil–Sen slope with its 95 % bounds and the Spearman rank correlation (the robust twin); and a permutation p of |r|
(`N_PERM_RANGE` random permutations of that range's years).

In [ ]:
sensitivity_df = rm.temperature_sensitivity(anomaly_df, temperature_df, n_perm=N_PERM_RANGE, seed=SEED)
sensitivity_df.describe()

## 6. The expected sensitivity from the annual temperature cycle (Evan & Eisenman 2021)

Their idealized model (a sinusoidal annual temperature cycle with annual mean T0 and amplitude T1, snow accumulating
below and melting above a threshold Tm) gives two expectations, evaluated on the ERA5 0.7° T0 / T1 climatology and
summarised over each range by the median of the cells its polygon touches (`rm.evan_eisenman_fields`):
the **melt-onset form**, the shift of the date the cycle crosses Tm upward, −(1/ω)/√(T1² − (T0 − Tm)²), which is the
like-for-like expectation for runoff onset; and their **eq. 2**, the shift of the snow-disappearance date, which is the
same times (1 + Ra/Rm) because a warmer year also has a shorter accumulation season. Tm = 0.18 °C and Ra/Rm = 0.34 are
their western-US station values; the melt-onset form does not depend on Ra/Rm. Undefined where the cycle never reaches
the threshold.

In [ ]:
range_geometries = gmba_gdf.set_index('GMBA_V2_ID').geometry.reindex(metrics_df['GMBA_V2_ID'].values).set_axis(metrics_df.index)
if ee_clim_ds is not None:
    ee_df = rm.evan_eisenman_expected(range_geometries.dropna(), ee_clim_ds).reindex(metrics_df.index)
else:
    ee_df = pd.DataFrame(index=metrics_df.index, columns=['ee_n_cells', 'ee_defined', 'ee_expected_sensitivity', 'ee_expected_onset_sensitivity'], dtype=float)
ee_df.describe()

## 7. The wide table, its dictionary and the per-band tables

In [ ]:
identity_cols = ['GMBA_V2_ID', 'continent', 'centroid_latitude', 'centroid_longitude']
timing_cols = ['mean_median_onset_dowy', 'mean_mad_days', 'lapse_rate_weighted_bins_per_100m', 'lapse_rate_weighted_bins_r2',
               'snowmelt_lapse_rate_per_100m', 'snowmelt_lapse_rate_corr']
per_year = pd.concat([anomaly_df.add_prefix('runoff_onset_anomaly_WY'),
                      coverage_df.add_prefix('coverage_WY'),
                      (coverage_df < rm.LOW_COVERAGE).where(anomaly_df.notna()).add_prefix('low_coverage_WY'),
                      temperature_df.add_prefix('spring_temperature_anomaly_WY'),
                      sunny_shady_year_df.add_prefix('sunny_shady_d_WY'),
                      lapse_year_df.add_prefix('lapse_rate_per_100m_WY')], axis=1)
statistics_df = pd.concat([metrics_df[identity_cols + timing_cols], snow_df, sunny_shady_df,
                           interannual_df, covariation_df, sensitivity_df, ee_df, per_year], axis=1)
statistics_df['abs_latitude_band'] = rm.by_latitude_band(statistics_df['centroid_latitude'].abs()).astype(str)
statistics_df.index.name = 'name'
statistics_df

In [ ]:
DICTIONARY = {
    'GMBA_V2_ID': ('', 'GMBA Inventory v2.0 standard-300 id'),
    'continent': ('', 'USGS continent nearest to the range centroid (Australia folded into Oceania)'),
    'centroid_latitude': ('degrees', 'latitude of the GMBA polygon centroid'),
    'centroid_longitude': ('degrees', 'longitude of the GMBA polygon centroid'),
    'mean_median_onset_dowy': ('day of water year', 'pixel-weighted mean of the 11-year median runoff onset (bins >= 100 px)'),
    'mean_mad_days': ('days', 'pixel-weighted mean of the per-pixel median absolute deviation of onset'),
    'lapse_rate_weighted_bins_per_100m': ('days per 100 m', 'sqrt(count)-weighted regression of bin median onset on elevation over (elevation, aspect) bins'),
    'lapse_rate_weighted_bins_r2': ('', 'r² of that regression'),
    'snowmelt_lapse_rate_per_100m': ('days per 100 m', 'regression of the aspect-collapsed elevation profile on elevation'),
    'snowmelt_lapse_rate_corr': ('', 'Pearson r of that regression'),
    'snow_km2': ('km²', 'mapped snow: pixels with an 11-year median onset, FCF <= 50 % (the analysis footprint; flat pixels included)'),
    'snow_under_forest_km2': ('km²', 'pixels with an 11-year median onset under forest (FCF > 50 %)'),
    'terrain_km2': ('km²', 'land pixels with a DEM (not ESA WorldCover water), snow or not, any forest cover'),
    'snow_pct_of_terrain': ('%', 'snow_km2 / terrain_km2'),
    'polygon_km2': ('km²', 'GMBA polygon area (EPSG:6933)'),
    'snow_pct_of_polygon': ('%', 'snow_km2 / polygon_km2: the inclusion quantity'),
    'included': ('', f'has metrics and snow_pct_of_polygon >= {INCLUSION_SNOW_PCT} %'),
    'sunny_shady_d': ('days', '2A of the cosine fit of elevation-removed onset against aspect (11-year median fields)'),
    'earliest_aspect_deg': ('degrees from north', 'aspect of the earliest onset in that fit'),
    'aspect_fit_r2': ('', 'r² of the cosine fit'),
    'n_years': ('', 'water years with a range-year anomaly'),
    'anomaly_sd': ('days', 'interannual s.d. of the range-year anomaly (>= 8 years)'),
    'anomaly_spread': ('days', 'max - min range-year anomaly (>= 8 years)'),
    'earliest_year': ('water year', 'year of the most negative anomaly (>= 8 years)'),
    'latest_year': ('water year', 'year of the most positive anomaly (>= 8 years)'),
    'topography_n_years': ('', 'years with both a per-year lapse rate and a per-year sunny-shady difference'),
    'lapse_rate_sd': ('days per 100 m', 'interannual s.d. of the per-year lapse rate (>= 8 years)'),
    'sunny_shady_sd': ('days', 'interannual s.d. of the per-year sunny-shady difference (>= 8 years)'),
    'lapse_aspect_r': ('', 'Pearson r across years between the per-year lapse rate and sunny-shady difference'),
    'lapse_aspect_p': ('', 'two-sided p of that r'),
    'sensitivity_n': ('', 'paired years in the temperature-sensitivity fit'),
    'ols_slope': ('days per °C', 'OLS slope of the range-year anomaly on the spring 2 m temperature anomaly'),
    'ols_se': ('days per °C', 'standard error of the OLS slope'),
    'ols_ci_low': ('days per °C', 'lower t-based 95 % bound of the OLS slope'),
    'ols_ci_high': ('days per °C', 'upper t-based 95 % bound of the OLS slope'),
    'pearson_r': ('', 'Pearson r of the fit'),
    'ols_p': ('', 'two-sided p of the OLS slope'),
    'theil_sen_slope': ('days per °C', 'Theil-Sen slope (median of pairwise slopes)'),
    'theil_sen_low': ('days per °C', 'lower 95 % bound of the Theil-Sen slope'),
    'theil_sen_high': ('days per °C', 'upper 95 % bound of the Theil-Sen slope'),
    'spearman_rho': ('', 'Spearman rank correlation'),
    'spearman_p': ('', 'two-sided p of rho'),
    'permutation_p': ('', f'share of {N_PERM_RANGE} permutations of the years with |r| >= the observed |r|'),
    'ee_n_cells': ('', 'ERA5 0.7° cells touched by the range polygon where the annual cycle crosses the melt threshold'),
    'ee_defined': ('', 'at least 3 such cells'),
    'ee_expected_sensitivity': ('days per °C', 'Evan & Eisenman (2021) eq. 2 (snow-disappearance date), median over the cells'),
    'ee_expected_onset_sensitivity': ('days per °C', 'the same model, melt-onset date (eq. 2 without the accumulation factor 1 + Ra/Rm), median over the cells'),
    'abs_latitude_band': ('degrees', 'reporting band of |centroid latitude|'),
}
PER_YEAR = {'runoff_onset_anomaly_WY': ('days', 'pixel-weighted mean onset anomaly of the water year (coverage >= 10 %)'),
            'coverage_WY': ('', "share of the range's median-onset pixels with a valid onset that year"),
            'low_coverage_WY': ('', f'the year has an anomaly but coverage < {rm.LOW_COVERAGE:.0%}'),
            'spring_temperature_anomaly_WY': ('°C', 'mean spring 2 m temperature anomaly of the year (ERA5-Land zonal mean)'),
            'sunny_shady_d_WY': ('days', 'sunny-shady difference (2A) of the year'),
            'lapse_rate_per_100m_WY': ('days per 100 m', 'lapse rate of the year')}
rows = []
for col in statistics_df.columns:
    prefix = next((p for p in PER_YEAR if col.startswith(p)), None)
    unit, meaning = PER_YEAR[prefix] if prefix else DICTIONARY[col]
    rows.append({'column': col, 'unit': unit, 'description': meaning})
dictionary_df = pd.DataFrame(rows)
assert dictionary_df['description'].notna().all()
dictionary_df

In [ ]:
statistics_path = results_dir / 'mountain_range_statistics.csv'
statistics_df.reset_index().assign(**results.provenance(config)).to_csv(statistics_path, index=False, float_format='%.4g')
dictionary_df.to_csv(results_dir / 'mountain_range_statistics_dictionary.csv', index=False)
print(f'{len(statistics_df)} ranges x {statistics_df.shape[1]} columns -> {statistics_path}')

In [ ]:
hypsometry_df.assign(**results.provenance(config)).to_csv(results_dir / 'mountain_range_hypsometry.csv', index=False, float_format='%.4g')
print(f"{hypsometry_df['name'].nunique()} ranges x {len(hypsometry_df)} bands -> mountain_range_hypsometry.csv")

In [ ]:
# long table (range, water year): the per-year topographic controls (sunny-shady difference, its earliest-onset aspect,
# lapse rate) and the 11-year median fields as water_year = 'median'
yearly_topography_df = pd.concat({'sunny_shady_d': sunny_shady_year_df.stack(dropna=False), 'earliest_aspect_deg': earliest_aspect_year_df.stack(dropna=False),
                                  'lapse_rate_per_100m': lapse_year_df.stack(dropna=False)}, axis=1)
yearly_topography_df.index.names = ['name', 'water_year']
yearly_topography_df = yearly_topography_df.dropna(how='all').reset_index().astype({'water_year': str})
median_rows_df = pd.DataFrame({'name': sunny_shady_df.index, 'water_year': 'median', 'sunny_shady_d': sunny_shady_df['sunny_shady_d'].values,
                               'earliest_aspect_deg': sunny_shady_df['earliest_aspect_deg'].values})
yearly_topography_df = pd.concat([median_rows_df, yearly_topography_df], ignore_index=True)
yearly_topography_df.assign(**results.provenance(config)).to_csv(results_dir / 'mountain_range_yearly_topography.csv', index=False, float_format='%.4g')
print(f'{len(yearly_topography_df)} rows -> mountain_range_yearly_topography.csv')

In [ ]:
band_aspect_df = rm.band_aspect(view_ds)
band_aspect_df.assign(**results.provenance(config)).to_csv(results_dir / 'mountain_range_band_aspect.csv', index=False, float_format='%.4g')
print(f"{len(band_aspect_df)} band rows ({(band_aspect_df['water_year'] == 'median').sum()} from the median fields)")
band_aspect_df.head()

## 8. Across-range summary

Counts and medians over the included ranges (and over the subset with a sensitivity fit), plus two permutation
nulls: the shared-year null for the number of significantly negative ranges (one permutation of the temperature
series' water-year labels for every range, which keeps the spatial dependence between ranges) and the within-range
shuffle null for the lapse–aspect co-variation.

In [ ]:
def q(s, qs):
    return [round(float(v), 2) for v in s.quantile(qs)]

def summarise(df):
    fit = df[df['ols_slope'].notna()]
    return {
        'n_ranges': int(len(df)), 'n_with_fit': int(len(fit)),
        'ols': {'pct_negative': round(100 * float((fit['ols_slope'] < 0).mean()), 1),
                'sig_negative': int(((fit['ols_p'] < 0.05) & (fit['ols_slope'] < 0)).sum()),
                'sig_positive': int(((fit['ols_p'] < 0.05) & (fit['ols_slope'] > 0)).sum()),
                'median_slope': round(float(fit['ols_slope'].median()), 2), 'slope_iqr': q(fit['ols_slope'], [.25, .75]),
                'median_r': round(float(fit['pearson_r'].median()), 2),
                'ci_below_zero': int((fit['ols_ci_high'] < 0).sum()),
                'permutation_sig_negative': int(((fit['permutation_p'] < 0.05) & (fit['ols_slope'] < 0)).sum())},
        'theil_sen': {'pct_negative': round(100 * float((fit['theil_sen_slope'] < 0).mean()), 1),
                      'ci_below_zero': int((fit['theil_sen_high'] < 0).sum()), 'ci_above_zero': int((fit['theil_sen_low'] > 0).sum()),
                      'median_slope': round(float(fit['theil_sen_slope'].median()), 2),
                      'median_spearman_rho': round(float(fit['spearman_rho'].median()), 2)},
        'median_onset_dowy': round(float(df['mean_median_onset_dowy'].median()), 1),
        'median_mad_days': round(float(df['mean_mad_days'].median()), 1),
        'median_lapse_rate_per_100m': round(float(df['lapse_rate_weighted_bins_per_100m'].median()), 2),
        'lapse_rate_iqr': q(df['lapse_rate_weighted_bins_per_100m'], [.25, .75]),
        'median_sunny_shady_d': round(float(df['sunny_shady_d'].median()), 1),
        'median_anomaly_sd': round(float(df['anomaly_sd'].median()), 1),
    }

included_df = statistics_df[statistics_df['included']]
fit_df = included_df[included_df['ols_slope'].notna()]
summary = {'dataset_version': VERSION, 'inclusion_snow_pct': INCLUSION_SNOW_PCT,
           'n_ranges_with_metrics': int(has_metrics.sum()), 'n_excluded': int((has_metrics & ~statistics_df['included']).sum()),
           **summarise(included_df)}
summary

In [ ]:
# by |latitude| band, the ranges' timing, topography and sensitivity
band_df = included_df.groupby('abs_latitude_band', observed=True).agg(
    n=('ols_slope', 'size'), median_slope=('ols_slope', 'median'),
    pct_sig=('ols_p', lambda p: 100 * (p < 0.05).mean()), median_lapse=('lapse_rate_weighted_bins_per_100m', 'median'),
    median_sunny_shady=('sunny_shady_d', 'median'), median_onset=('mean_median_onset_dowy', 'median'))
summary['by_abs_latitude_band'] = band_df.round(2).to_dict(orient='index')
band_df.round(2)

In [ ]:
# extremes and distributions quoted in the text
summary['onset_p5_p95_dowy'] = q(included_df['mean_median_onset_dowy'], [.05, .95])
summary['lapse_rate_p5_p95'] = q(included_df['lapse_rate_weighted_bins_per_100m'], [.05, .95])
summary['sunny_shady_p5_p95'] = q(included_df['sunny_shady_d'], [.05, .95])
summary['anomaly_sd_iqr'] = q(included_df['anomaly_sd'].dropna(), [.25, .75])
summary['median_anomaly_spread'] = round(float(included_df['anomaly_spread'].median()), 1)
summary['n_ranges_ge8_years'] = int(included_df['anomaly_sd'].notna().sum())
summary['earliest_year_counts'] = {int(k): int(v) for k, v in included_df['earliest_year'].dropna().astype(int).value_counts().sort_index().items()}
summary['latest_year_counts'] = {int(k): int(v) for k, v in included_df['latest_year'].dropna().astype(int).value_counts().sort_index().items()}
summary['positive_significant'] = fit_df[(fit_df['ols_p'] < 0.05) & (fit_df['ols_slope'] > 0)][['ols_slope', 'pearson_r']].round(2).reset_index().values.tolist()
summary['interannual_topography'] = {
    'n_ranges': int(included_df['lapse_rate_sd'].notna().sum()),
    'median_lapse_rate_sd': round(float(included_df['lapse_rate_sd'].median()), 2),
    'median_sunny_shady_sd': round(float(included_df['sunny_shady_sd'].median()), 2),
    'lapse_rate_cv_pct': round(float((included_df['lapse_rate_sd'] / included_df['lapse_rate_weighted_bins_per_100m'].abs()).median() * 100)),
    'sunny_shady_cv_pct': round(float((included_df['sunny_shady_sd'] / included_df['sunny_shady_d']).median() * 100))}
summary['interannual_topography']

In [ ]:
field_significance = rm.field_significance(anomaly_df, temperature_df, fit_df.index, n_perm=N_PERM_FIELD, seed=SEED)
summary['field_significance'] = {k: (round(v, 3) if isinstance(v, float) else v) for k, v in field_significance.items()}
summary['field_significance']

In [ ]:
covariation_null = rm.covariation_null(lapse_year_df, sunny_shady_year_df, included_df.index, n_perm=N_PERM_COVARIATION, seed=SEED)
summary['lapse_aspect_covariation'] = {k: (round(v, 3) if isinstance(v, float) else v) for k, v in covariation_null.items()}
summary['lapse_aspect_covariation']

In [ ]:
ee_fit_df = fit_df[fit_df['ee_defined'].fillna(False).astype(bool) & (fit_df['sensitivity_n'] >= 8)]
if len(ee_fit_df):
    def compare(col):
        return {'n': int(len(ee_fit_df)), 'spearman_rho': round(float(ee_fit_df[[col, 'ols_slope']].corr(method='spearman').iloc[0, 1]), 2),
                'pearson_r': round(float(ee_fit_df[[col, 'ols_slope']].corr().iloc[0, 1]), 2),
                'median_expected': round(float(ee_fit_df[col].median()), 1), 'median_observed': round(float(ee_fit_df['ols_slope'].median()), 1),
                'frac_observed_weaker': round(float((ee_fit_df['ols_slope'] > ee_fit_df[col]).mean()), 2),
                'by_abs_latitude_band': ee_fit_df.groupby('abs_latitude_band').agg(n=('ols_slope', 'size'), observed=('ols_slope', 'median'),
                                                                                  expected=(col, 'median')).round(1).to_dict(orient='index')}
    summary['evan_eisenman'] = {'melt_onset_form': compare('ee_expected_onset_sensitivity'), 'eq2_disappearance': compare('ee_expected_sensitivity')}
summary.get('evan_eisenman')

In [ ]:
# the seasonal arch: each range's largest band amplitude (2A) in the median fields
arch_df = band_aspect_df[band_aspect_df['water_year'] == 'median']
arch_df = arch_df[arch_df['name'].isin(included_df.index)]
summary['band_peak_sunny_shady_d'] = {'median': round(float((2 * arch_df.groupby('name')['amp_d'].max()).median()), 1)}
# robustness: the summary numbers under other inclusion thresholds
robust_rows = []
for rule in ROBUSTNESS_RULES:
    sub = statistics_df[has_metrics & (statistics_df['snow_pct_of_polygon'] >= rule)]
    s = summarise(sub)
    robust_rows.append({'rule_pct': rule, 'n_ranges': s['n_with_fit'], 'pct_negative': s['ols']['pct_negative'],
                        'sig_negative': s['ols']['sig_negative'], 'sig_positive': s['ols']['sig_positive'],
                        'median_slope': s['ols']['median_slope'], 'median_r': s['ols']['median_r'],
                        'theil_sen_ci_below_zero': s['theil_sen']['ci_below_zero'], 'median_lapse_rate': s['median_lapse_rate_per_100m'],
                        'median_sunny_shady_d': s['median_sunny_shady_d']})
robustness_df = pd.DataFrame(robust_rows)
robustness_df.assign(**results.provenance(config)).to_csv(results_dir / 'mountain_range_inclusion_robustness.csv', index=False)
robustness_df

In [ ]:
summary['_provenance'] = results.provenance(config)
with open(results_dir / 'mountain_range_summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
print(json.dumps({k: summary[k] for k in ('n_ranges', 'n_with_fit', 'ols', 'theil_sen')}, indent=1))